In [ ]:
import pandas as pd
from pathlib import Path 
import geopandas as geopd
import plotly.graph_objs as go 
import plotly.figure_factory as ff
import plotly.express as px  
import matplotlib.pyplot as plt 
import numpy as np 

from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import squareform
from sklearn.metrics import silhouette_score
from scipy.signal import find_peaks

import sys

REPO_ROOT = Path.cwd() if (Path.cwd() / "utils.py").exists() else Path.cwd().parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
    
from utils import get_s3, get_mapping
from aixpa_analisi_indici.help_plots import plot_stats, plot_overlapped

# CARICAMENTO DATI 

In [ ]:
vodafone_attendences_new = pd.read_csv(get_s3("vodafone_attendences_new.csv"))

geojson_comuni_json_data = geopd.read_file(get_s3("TRENTINO-comuni_Vodafone_2023.geojson"))
json_vodafone = get_mapping("mapping_comuni_into_vodafone_Trento.json")
json_apt = get_mapping("map_comuni_into_apt.json")

In [ ]:
location_map = dict(
    zip(
        geojson_comuni_json_data["id"].astype(str),
        geojson_comuni_json_data["name"].str.upper(),
    )
)

vodafone_attendences_new['date'] = pd.to_datetime(vodafone_attendences_new['date'])
vodafone_attendences_new = vodafone_attendences_new.sort_values('date').reset_index(drop=True)

vodafone_attendences_merged_comuni = vodafone_attendences_new[
    vodafone_attendences_new["locType"] == "TN_MKT_AL_3"
]
vodafone_attendences_merged_comuni["comune"] = vodafone_attendences_merged_comuni["locId"].astype(str).map(location_map)
vodafone_attendences_merged_comuni["ID_COMUNE"] = vodafone_attendences_merged_comuni["comune"].map(json_vodafone)
# Fix vigo and pozza 
mask = vodafone_attendences_merged_comuni["comune"].isin(["VIGO DI FASSA", "POZZA DI FASSA"])
vodafone_attendences_merged_comuni.loc[mask, "comune"] = "SAN GIOVANNI DI FASSA" 
vodafone_attendences_merged_comuni.loc[mask, "ID_COMUNE"] = pd.Series(
    [[22250]] * mask.sum(), index=vodafone_attendences_merged_comuni.index[mask]  
)

tourists_attendences = vodafone_attendences_merged_comuni[
    vodafone_attendences_merged_comuni['userProfile'] == 'TOURIST'
].groupby(['date', 'comune']).agg({'value': 'sum', 'ID_COMUNE':'first'}).reset_index()

In [ ]:
pq_tourists = pd.read_parquet("tourists_seasonal.parquet")
pq_tourists.sort_values(by = 'date')

In [ ]:
pq_tourists = pq_tourists[pq_tourists['has_seasonality']]

### clusters (solo quelli con stagionalita')

In [ ]:
fig = plot_overlapped(pq_tourists)
fig.show()

In [ ]:
pv=pq_tourists.pivot(index="date", columns="comune", values="value")
s = pv.isna().sum()
s[s>0]

In [ ]:
min_vals = pv.min()
max_vals = pv.max()

# 2. NORMALIZZAZIONE (Fatti in anticipo su tutti i comuni)
ts_normalized = (pv - min_vals) / (max_vals - min_vals).replace(0, np.nan)
# ts_normalized = ts_normalized.fillna(0)

# CORRELAZIONE 
corr_matrix = ts_normalized.corr()
corr_matrix

In [ ]:
DISTANCE = 60
CLUSTERS = 2
METHOD = "ward"
FORMAT_SAVE = "parquet"

def ward_distance(corr_matrix):
    return np.sqrt(2 * (1 - corr_matrix)) 

In [ ]:
dist_matrix = ward_distance(corr_matrix) if METHOD == "ward" else 1 - corr_matrix
condensed_dist = squareform(dist_matrix.values, checks=False)
Z = linkage(condensed_dist, method=METHOD)

print("--- Confronto Silhouette Score ---")
for method in ["average", "complete", "single", "ward"]:
    Z_test = linkage(condensed_dist, method=method)
    
    for k in range(2, 8):
        labels = fcluster(Z_test, t=k, criterion="maxclust")
        # Calcolo del silhouette score usando la matrice delle distanze precalcolata
        score = silhouette_score(dist_matrix, labels, metric="precomputed")
        print(f"Metodo: {method:8s} | k={k} | Silhouette: {score:.3f}")

from scipy.cluster.hierarchy import fcluster
# (supponendo di avere la linkage matrix 'Z')
clusters = fcluster(Z, t=4, criterion='maxclust')
pd.Series(clusters).value_counts()

In [ ]:
dendogram = ff.create_dendrogram(
    dist_matrix.values,
    labels=corr_matrix.columns.tolist(),
    distfun=lambda x: condensed_dist,
    linkagefun=lambda d: Z,
)


dendogram.update_layout(
    title=f"Dendrogramma della Similarità Turistica (su dati Rolling + Normalizzati), metodo {METHOD}",
    xaxis_title="Comuni",
    yaxis_title="Distanza",
    width=2000,
    height=700,
    template="plotly_white"
)
dendogram.update_xaxes(tickangle=-45)
dendogram.show()

clusters = fcluster(Z, t=CLUSTERS, criterion='maxclust')

df_clusters = pd.DataFrame({
    'comune': corr_matrix.columns,
    'cluster': clusters
}).sort_values('cluster')

print(f"--- Distribuzione Comuni nei {CLUSTERS} Cluster ---")
print(df_clusters['cluster'].value_counts())

In [ ]:
gdf_map = geojson_comuni_json_data.copy()
gdf_map["comune_upper"] = gdf_map["name"].str.upper()

# Fix per Vigo e Pozza di Fassa -> San Giovanni di Fassa
mask_fassa = gdf_map["comune_upper"].isin(["VIGO DI FASSA", "POZZA DI FASSA"])
gdf_map.loc[mask_fassa, "comune_upper"] = "SAN GIOVANNI DI FASSA"

map_df = gdf_map.merge(
    df_clusters, 
    left_on="comune_upper", 
    right_on="comune", 
    how="left"
)

map_df["cluster_str"] = map_df["cluster"].astype(str)

fig_map = px.choropleth(
    map_df,
    geojson=map_df.__geo_interface__,
    locations=map_df.index,
    color="cluster_str",
    hover_name="name",  # Mostra il nome originale del poligono al passaggio del mouse
    hover_data={"comune_upper": True, "cluster": True},
    title=f"Mappa dei Cluster Turistici in Trentino (Metodo {METHOD}, k={CLUSTERS})",
    color_discrete_sequence=px.colors.qualitative.Bold,
    template="plotly_white"
)

fig_map.update_geos(fitbounds="locations", visible=False)
fig_map.update_layout(
    width=1000,
    height=800,
    margin={"r": 0, "t": 50, "l": 0, "b": 0},
    legend_title="Cluster"
)

fig_map.show()

In [ ]:
all_counts = [] 
show_peaks = False 

for cc in df_clusters['cluster'].unique():
    clust_subdf = df_clusters[df_clusters['cluster'] == cc]
    comuni_cluster = clust_subdf['comune'].unique()
    
    ts_normalized_clust = ts_normalized.loc[:, comuni_cluster].copy()

    mean_ts = ts_normalized_clust.mean(axis=1)
    std_ts = ts_normalized_clust.std(axis=1)
    
    upper_bound = (mean_ts + std_ts).clip(upper=1.0)
    lower_bound = (mean_ts - std_ts).clip(lower=0.0)

    ts_normalized_long = (
        ts_normalized_clust
        .reset_index()
        .rename(columns={"index": "date"})
        .melt(id_vars="date", var_name="comune", value_name="value")
        .dropna()
    )
    
    # Inizializziamo la figura
    fig = plot_stats(ts_normalized_long)

    # --- Banda di Deviazione Standard con Contornini Eleganti ---
    border_style = dict(width=1.2, color="rgba(210, 140, 0, 0.7)", dash="dash")

    # Banda superiore
    fig.add_trace(go.Scatter(
        x=mean_ts.index,
        y=upper_bound,
        mode='lines',
        line=border_style,
        showlegend=False,
        hoverinfo='skip'
    ))

    # Banda inferiore con riempimento (tonexty)
    fig.add_trace(go.Scatter(
        x=mean_ts.index,
        y=lower_bound,
        mode='lines',
        line=border_style,
        fill='tonexty',
        fillcolor="rgba(255, 200, 10, 0.25)",
        name='Deviazione Std',
        showlegend=True,
        hoverinfo='skip'
    ))

    # Linea della media del cluster
    fig.add_trace(go.Scatter(
        x=mean_ts.index,
        y=mean_ts,
        mode='lines',
        line=dict(color='rgba(31, 119, 180, 1)', width=2.5),
        name='Media Cluster'
    ))

    # --- AGGIUNTA: Titolo dinamico per ogni cluster ---
    num_comuni = len(comuni_cluster)
    fig.update_layout(
        title=f"Profilo Stagionale - Cluster {cc} (Comuni inclusi: {num_comuni})"
    )

    fig.show()

## INDICE DI GINI

In [ ]:
def gini_index(array):
    """Calcola l'indice di Gini per una serie temporale di presenze (es. 365 giorni)."""
    array = np.asarray(array, dtype=np.float64)

    if np.all(array == 0) or len(array) == 0 or np.sum(array) == 0:
        return 0.0

    n = len(np.sort(array))
    index = np.arange(1, n + 1)

    return (2 * np.sum(index * array)) / (n * np.sum(array)) - (n + 1) / n


# --- APPLICAZIONE SUI TUOI DATI ---

df_full = (
    tourists_attendences.pivot(index="date", columns="comune", values="value")
    .reindex(pd.date_range("2022-01-01", "2025-12-31", freq="D"))
    .fillna(0)
    .unstack()
    .reset_index()
)
df_full.columns = ["comune", "date", "value"]

df_full["year"] = df_full["date"].dt.year

df_gini = (
    df_full.groupby(["comune", "year"])["value"]
    .apply(gini_index)
    .reset_index()
    .rename(columns={"value": "gini_seasonality"})
)

# Visualizza i comuni con maggiore/minore stagionalità nel 2023
print("Comuni a più alta stagionalità (2023):")
print(
    df_gini[df_gini["year"] == 2023]
    .sort_values("gini_seasonality", ascending=False)
    .head(10)
)

### ESEMPIO TIME SERIES Gini 

In [ ]:
comune_test = "ALA"
year_test = 2022
ts_comune = df_full[(df_full['comune'] == comune_test) & (df_full['year'] == year_test)]

# ts_rolling_14 = ts_comune.rolling(window=14, center=True).mean()
fig_smooth = go.Figure()

fig_smooth.add_trace(go.Scatter(
    x=ts_comune['date'], y=ts_comune['value'],
    mode='lines', name='Dati Grezzi (Giornalieri)',
    line=dict(color="#676867", width=1)
))

fig_smooth.update_layout(
    title=f"Smoothing - Comune di {comune_test}",
    xaxis_title="Data",
    yaxis_title="Presenze Turistiche",
    template="plotly_white",
    hovermode="x unified"
)
fig_smooth.show()

In [ ]:
## 
YEARS = [2023, 2024, 2025]   # controlla prima con s.index.min() / s.index.max()

s = tourists_attendences.pivot(index="date", columns="comune", values="value")
print(s.index.min(), s.index.max())

# solo anni completi; i giorni mancanti restano NaN (NIENTE fillna(0))
s = s.reindex(pd.date_range(f"{YEARS[0]}-01-01", f"{YEARS[-1]}-12-31", freq="D"))

grp = [s.index.year, s.index.month]
monthly = s.groupby(grp).mean()
coverage = s.notna().groupby(grp).mean()
monthly.index.names = coverage.index.names = ["year", "month"]

# scarta i mesi con meno dell'80% dei giorni disponibili
monthly = monthly.where(coverage >= 0.8)

def gini(x, normalize=True):
    x = np.asarray(x, dtype=float)
    if np.isnan(x).any() or x.sum() == 0:      # anno incompleto o comune vuoto
        return np.nan
    x = np.sort(x); n = len(x)
    g = (2*np.arange(1, n+1) - n - 1).dot(x) / (n * x.sum())
    return g * n/(n-1) if normalize else g     # normalize -> scala 0-1

gini_year = monthly.groupby(level="year").agg(lambda c: gini(c.values))
# righe = anno, colonne = comune

gini_comune = pd.DataFrame({
    "gini_mean": gini_year.mean(),
    "gini_std":  gini_year.std(),      # stabilità tra anni
    "n_years":   gini_year.notna().sum(),
    "presenze_medie_giorno": s.mean(),  # per segnalare i comuni piccoli
})

res = df_clusters.set_index("comune").join(gini_comune)

print(res.groupby("cluster")["gini_mean"].describe())

fig = px.box(
    res.reset_index(), x="cluster", y="gini_mean", points="all",
    hover_name="comune", color=res["cluster"].astype(str).values,
    title="Indice di Gini mensile per cluster", template="plotly_white",
)
fig.show()
map_gini = gdf_map.merge(res.reset_index(), left_on="comune_upper", right_on="comune", how="left")
fig_g = px.choropleth(
    map_gini, geojson=map_gini.__geo_interface__, locations=map_gini.index,
    color="gini_mean", color_continuous_scale="YlOrRd", hover_name="name",
    title="Gini della stagionalità mensile", template="plotly_white",
)
fig_g.update_geos(fitbounds="locations", visible=False)
fig_g.show()

## CLUSTER E INDICE DI GINI

Uniamo cluster e indice di Gini in un unico dataframe (`comune`, `cluster`, `gini`):
- i comuni senza stagionalità, esclusi dal clustering, formano un gruppo a sé (`cluster = 0`);
- `gini` è il `gini_mean` calcolato sopra (Gini mensile, media sugli anni, scala 0-1).

Nella mappa il colore indica il cluster e l'opacità di ogni comune è pari al suo indice di Gini.

In [ ]:
NON_SEASONAL_CLUSTER = 0

# flag di stagionalità per tutti i comuni (pq_tourists sopra è già filtrato)
has_seasonality = (
    pd.read_parquet("tourists_seasonal.parquet")
    .groupby("comune")["has_seasonality"]
    .first()
)
assert set(df_clusters["comune"]) == set(has_seasonality[has_seasonality].index)

# i comuni esclusi dal clustering (senza stagionalità) formano un gruppo a sé
df_comuni_cluster = (
    has_seasonality.index.to_frame(index=False)
    .merge(df_clusters, on="comune", how="left")
)
df_comuni_cluster["cluster"] = (
    df_comuni_cluster["cluster"].fillna(NON_SEASONAL_CLUSTER).astype(int)
)

df_comuni_cluster["cluster"].value_counts().sort_index()

In [ ]:
# gini_mean della sezione precedente: Gini dei valori mensili, media sugli anni (scala 0-1)
df_cluster_gini = df_comuni_cluster.join(
    gini_comune["gini_mean"].rename("gini"), on="comune"
)

print("Comuni senza indice di Gini:",
      df_cluster_gini.loc[df_cluster_gini["gini"].isna(), "comune"].tolist())
print(df_cluster_gini.groupby("cluster")["gini"].describe())
df_cluster_gini

In [ ]:
cluster_rgb = {1: "42, 120, 214", 2: "235, 104, 52", NON_SEASONAL_CLUSTER: "27, 175, 122"}

map_cg = gdf_map.merge(df_cluster_gini, left_on="comune_upper", right_on="comune")

fig_cg = go.Figure()
for c, rgb in cluster_rgb.items():
    sub = map_cg[map_cg["cluster"] == c]
    fig_cg.add_trace(go.Choropleth(
        geojson=sub.__geo_interface__,
        locations=sub.index,
        z=sub["gini"],
        zmin=0,
        zmax=1,
        colorscale=[[0, f"rgba({rgb}, 0)"], [1, f"rgba({rgb}, 1)"]],  # opacità = Gini
        showscale=False,
        text=sub["name"],
        hovertemplate=f"%{{text}}<br>Cluster {c}<br>Gini: %{{z:.2f}}<extra></extra>",
    ))

fig_cg.update_geos(fitbounds="locations", visible=False)
fig_cg.update_layout(
    title="Cluster turistici (colore) e indice di Gini (opacità)",
    width=1000,
    height=700,
    template="plotly_white",
)
fig_cg.show()

In [ ]:
fig_gini = go.Figure(go.Choropleth(
    geojson=map_cg.__geo_interface__,
    locations=map_cg.index,
    z=map_cg["gini"],
    zmin=0,
    zmax=1,
    colorscale = "Ice_r",
    # colorscale=[[0, "rgba(42, 120, 214, 0)"], [1, "rgba(42, 120, 214, 1)"]],  # opacità = Gini
    showscale=False,
    text=map_cg["name"],
    hovertemplate="%{text}<br>Gini: %{z:.2f}<extra></extra>",
))

fig_gini.update_geos(fitbounds="locations", visible=False)
fig_gini.update_layout(
    title="Indice di Gini per comune (opacità)",
    width=1000,
    height=700,
    template="plotly_white",
)
fig_gini.show()